### Setup & Loading Data

In [19]:
#Read/Manipulate CSV Files
import pandas as pd
import numpy as np
from tqdm import tqdm


In [20]:
#Load the CSV file into pandas
df = pd.read_csv(r'C:\Users\matth\OneDrive\Desktop\dataset\datasetA.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
df.head()

Dataset shape: (10500, 12)

Columns: ['title', 'link', 'date', 'source', 'number_of_characters_title', 'number_of_words_title', 'day_of_week', 'month', 'year', 'quarter', 'is_weekend', 'classes_str']


,title,link,date,source,number_of_characters_title,number_of_words_title,day_of_week,month,year,quarter,is_weekend,classes_str
0,Google’s AI is the ‘worst’ for stealing conten...,https://news.google.com/rss/articles/CBMipgFBV...,2025-09-11,Fortune,74,13,Thursday,September,2025,3,False,Sentiment (Positive / Negative Feelings); Huma...
1,Powering the Next Wave of Enterprise Innovatio...,https://news.google.com/rss/articles/CBMitgFBV...,2025-09-11,Silicon Canals,106,16,Thursday,September,2025,3,False,"Creativity, Expression & Identity; Work, Jobs ..."
2,AI a ‘strategic necessity’ law lecturer says,https://news.google.com/rss/articles/CBMiiAFBV...,2025-09-11,qlsproctor.com.au,64,9,Thursday,September,2025,3,False,"Society, Ethics & Culture"
3,Datacom sees AI agents as pivotal to legacy ap...,https://news.google.com/rss/articles/CBMirAFBV...,2025-09-11,ARNnet,70,12,Thursday,September,2025,3,False,"Routine, Lifestyle & Behavior"
4,"Student Blog: Startups, AI, and Lessons from S...",https://news.google.com/rss/articles/CBMijwFBV...,2025-09-11,The University of Queensland,85,13,Thursday,September,2025,3,False,"Learning, Knowledge & Education"


### Data Preprocessing

In [23]:
#Selecting Column
#TECHY!!! WORKS WITH TEXT ELSE TITLE >:)
text_col = 'text' if 'text' in df.columns else 'title'

#Cleaning Column
df[text_col] = df[text_col].fillna("").astype(str)

#Removing empty
df = df[df[text_col].str.strip() != ""]

print(f"Working with column: '{text_col}'")
print(f"Clean dataset: {len(df)} articles")

Working with column: 'title'
Clean dataset: 10500 articles


### Rule-Based Approach (VADER)

In [24]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

def get_vader_sentiment(text):
    #Returns Vader Compound Score (-1 to 1)
    scores = vader.polarity_scores(text)
    return scores['compound']

#Applying to dataset
print("Running VADER analysis...")
df['vader_score'] = df[text_col].apply(get_vader_sentiment)

#Categorization
def classify_sentiment(score):
    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    else:
        return 'neutral'

df['vader_label'] = df['vader_score'].apply(classify_sentiment)

print("\nVADER Results:")
print(df['vader_label'].value_counts())
print(f"Average score: {df['vader_score'].mean():.3f}")

Running VADER analysis...

VADER Results:
vader_label
positive    4783
neutral     3709
negative    2008
Name: count, dtype: int64
Average score: 0.137


### Transformer-Based (RoBERTa) Approach

In [ ]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    truncation=True
)

#Labeling the Labels e _ e
label_map = {
    "LABEL_0": "negative",
    "LABEL_1": "neutral",
    "LABEL_2": "positive"
}

Device set to use cpu


In [27]:
def get_roberta_sentiment(text, max_words=512):
    #Analyzes Sentiment & Chunks Text
    words = text.split()

    if not words:
        return 'neutral', 0.33

    #Proccess | Chunks
    chunks = [' '.join(words[i:i+max_words]) for i in range(0, len(words), max_words)]

    #Predictions | Each Chunk
    results = sentiment_model(chunks, batch_size=8, truncation=True)

    #Extract | Labels & Scores
    labels = [label_map[r['label']] for r in results]
    scores = [r['score'] for r in results]

    #Return | MCL and AVG Confidence
    from collections import Counter
    most_common = Counter(labels).most_common(1)[0][0]
    avg_conf = np.mean([s for l, s in zip(labels, scores) if l == most_common])

    return most_common, avg_conf

In [28]:
#Applying to dataset (subset rn for testing)
n_samples = 500 #CHANGE WHEN USING FULL DATASET

print(f"Running RoBERTa analysis on {n_samples} articles...")

results = []
for text in tqdm(df[text_col].head(n_samples)):
    label, conf = get_roberta_sentiment(text)
    results.append({'label' : label, 'confidence': conf})

results_df = pd.DataFrame(results)
df.loc[:n_samples-1, 'roberta_label'] = results_df['label']
df.loc[:n_samples-1, 'roberta_conf'] = results_df['confidence']

print("\nRoBERTa Results:")
print(df['roberta_label'].value_counts())
print(f"Average confidence: {df['roberta_conf'].mean():.3f}")

Running RoBERTa analysis on 500 articles...


100%|██████████| 500/500 [00:25<00:00, 19.65it/s]


RoBERTa Results:
roberta_label
neutral     362
positive     89
negative     49
Name: count, dtype: int64
Average confidence: 0.703


### Traditonal ML (TF-IDF + Logistic Regression) Approach

In [29]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

In [31]:
#Prep training data
train_data = df.dropna(subset=['roberta_label']).copy()

print(f"Training on {len(train_data)} RoBERTa-labeled articles")

#Split | Train & Test
X_train, X_test, y_train, y_test = train_test_split(
    train_data[text_col],
    train_data['roberta_label'],
    test_size=0.2,
    random_state=42,
    stratify=train_data['roberta_label']
)

print(f"Train size: {len(X_train)} | Test size: {len(X_test)}")

Training on 500 RoBERTa-labeled articles
Train size: 400 | Test size: 100


In [ ]:
#TF-IDF Features
print("\nCreating TF-IDF features...")

tfidf = TfidfVectorizer(
    max_features= 5000, # Top 5000 words
    ngram_range=(1,2), # Unigrams and bigrams [the, the dog]
    min_df =2,          # Minimum word appearance (on 2 docs)
    stop_words='english' # Remove common words
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f"Feature matrix shape: {X_train_tfidf.shape}")


Creating TF-IDF features...
Feature matrix shape: (400, 664)


In [34]:
#Training Logistic Regression
print("\nTraining Logisitic Regression...")

lr_model = LogisticRegression(
    max_iter=1000,
    random_state=42,
    class_weight='balanced' # Handles class imbalance
)

lr_model.fit(X_train_tfidf, y_train)

# Evaluation on test set
y_pred = lr_model.predict(X_test_tfidf)
print("\nTest Set Performance:")
print(classification_report(y_test, y_pred))


Training Logisitic Regression...

Test Set Performance:
              precision    recall  f1-score   support

    negative       0.33      0.30      0.32        10
     neutral       0.74      0.62      0.68        72
    positive       0.20      0.33      0.25        18

    accuracy                           0.54       100
   macro avg       0.42      0.42      0.41       100
weighted avg       0.60      0.54      0.56       100



In [36]:
# Applying to full dataset
print("\nApplying ML model to full dataset...")

X_full_tfidf = tfidf.transform(df[text_col])
df['ml_label'] = lr_model.predict(X_full_tfidf)
df['ml_confidence'] = lr_model.predict_proba(X_full_tfidf).max(axis=1)

print("\nML Model Results:")
print(df['ml_label'].value_counts())
print(f"Average confidence: {df['ml_confidence'].mean():.3f}")


Applying ML model to full dataset...

ML Model Results:
ml_label
neutral     7099
positive    1973
negative    1428
Name: count, dtype: int64
Average confidence: 0.519


### Comparison Between All Aproaches

In [37]:
#Comparison DataFrame (on subset with all labels)
comparison = df.dropna(subset=['roberta_label']).copy()

print(f"Comparing {len(comparison)} articles with all three methods \n")

# Agreement rates
vader_roberta = (comparison['vader_label'] == comparison['roberta_label']).mean()
vader_ml = (comparison['vader_label'] == comparison ['ml_label']).mean()
roberta_ml = (comparison['roberta_label'] == comparison['ml_label']).mean()

print("Agreement between methods:")
print(f" VADER <-> RoBERTa: {vader_roberta:.1%}")
print(f" VADER <-> ML: {vader_ml:.1%}")
print(f" RoBERTa <-> ML: {roberta_ml:.1%}")

Comparing 500 articles with all three methods 

Agreement between methods:
 VADER <-> RoBERTa: 49.8%
 VADER <-> ML: 45.4%
 RoBERTa <-> ML: 83.6%


In [39]:
# Method Disagreement | Examples
print("\nExamples where methods disagree:")
disagreements = comparison[
    (comparison['vader_label'] != comparison['roberta_label']) |
    (comparison['vader_label'] != comparison['ml_label'])
].head(5)

for idx, row in disagreements.iterrows():
    print(f"\nText: {row[text_col][:100]}...")
    print(f" Vader: {row['vader_label']} ({row['vader_score']:.2f})")
    print(f" RoBERTa: {row['roberta_label']} ({row['roberta_conf']:.2f})")
    print(f" ML: {row['ml_label']} ({row['ml_confidence']:.2f})")



Examples where methods disagree:

Text: AI a ‘strategic necessity’ law lecturer says...
 Vader: neutral (0.00)
 RoBERTa: neutral (0.82)
 ML: negative (0.50)

Text: Datacom sees AI agents as pivotal to legacy app modernisation...
 Vader: neutral (0.00)
 RoBERTa: neutral (0.62)
 ML: positive (0.57)

Text: Is Donald Trump's video on Charlie Kirk's shooting AI-generated? Internet questions authenticity...
 Vader: neutral (0.00)
 RoBERTa: neutral (0.59)
 ML: negative (0.42)

Text: Klarna 'course-correct' after aggressive AI adoption – reports...
 Vader: negative (-0.15)
 RoBERTa: neutral (0.79)
 ML: neutral (0.44)

Text: AI chatbot users report mental health issues...
 Vader: neutral (0.00)
 RoBERTa: negative (0.65)
 ML: negative (0.73)


### Saving Results

In [45]:
output_path = 'datasetA_with_sentiment.csv' #Change this to somewhere you want fr; saves to explorer
df.to_csv(output_path, index=False)
print(f"Saved results to: {output_path}")

#Stats Summary
print("\n" + "="*50)
print("FINAL SUMMARY")
print("="*50)
print(f"Total articles analyzed: {len(df)}")
print(f"\nMethod Coverage:")
print(f" VADER: {df['vader_label'].notna().sum()} articles")
print(f" RoBERTa: {df['roberta_label'].notna().sum()} articles")
print(f" ML: {df['ml_label'].notna().sum()} articles")

print(f"\nOverall Sentiment Distribution:")

print("\nVADER:")
vader_counts = df['vader_label'].value_counts().sort_index()
for label, count in vader_counts.items():
    pct = (count / len(df)) * 100
    print(f"  {label}: {count} ({pct:.1f}%)")

if 'roberta_label' in df.columns:
    print("\nRoBERTa:")
    roberta_counts = df['roberta_label'].value_counts().sort_index()
    roberta_total = df['roberta_label'].notna().sum()
    for label, count in roberta_counts.items():
        pct = (count / roberta_total) * 100
        print(f"  {label}: {count} ({pct:.1f}%)")

if 'ml_label' in df.columns:
    print("\nML Model:")
    ml_counts = df['ml_label'].value_counts().sort_index()
    for label, count in ml_counts.items():
        pct = (count / len(df)) * 100
        print(f"  {label}: {count} ({pct:.1f}%)")

Saved results to: datasetA_with_sentiment.csv

FINAL SUMMARY
Total articles analyzed: 10500

Method Coverage:
 VADER: 10500 articles
 RoBERTa: 500 articles
 ML: 10500 articles

Overall Sentiment Distribution:

VADER:
  negative: 2008 (19.1%)
  neutral: 3709 (35.3%)
  positive: 4783 (45.6%)

RoBERTa:
  negative: 49 (9.8%)
  neutral: 362 (72.4%)
  positive: 89 (17.8%)

ML Model:
  negative: 1428 (13.6%)
  neutral: 7099 (67.6%)
  positive: 1973 (18.8%)


### INFORMATION RELATED TO CODE

**FOR MANNY AND NOAH**

- All sentiment scores are saved in the CSV

- Column names: 'vader_label', 'roberta_label', 'ml_label'

- Able to merge with topic model using article ID

**TO PROCESS MORE WITHIN DATASET**

- Change 'n_snamples = 500' to 'n_samples = len(df)' in RoBERTa section

- ^^ Forsure takes longer x _ x

**RELATED TO SCRAPED DATA**

- Wanna actually use web-scraped data..? just change the file path in "Setup & Loading Data"

- Pipleine is functional with any CSV with text or title column

